In [1]:
import os, sys, shutil
import numpy as np
import pandas as sp
import pandas as pd
import nibabel as nib
import json
import wget

In [2]:
home = os.path.expanduser("~")
print(home)
atlas_dir = os.path.join(home, 'research_projects/snaplab_tools/data/atlases')

/home/lindenmp


## MNI

In [3]:
# MNIvolumetric
in_dir = os.path.join(home, 'MSA/subcortex/Group-Parcellation/3T/Cortex-Subcortex/MNIvolumetric')
n_regions = 100
msa_max_indices = [16, 32, 50, 54]

# MNI152NLin2009cAsym
for tian_scale in [1, 2, 3, 4]:
    out_dir = os.path.join(atlas_dir, 'MSA', 'atlas-MSA{0}'.format(tian_scale))
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    for res in [1, 2]:
        shutil.copyfile(
            os.path.join(in_dir, 'Schaefer2018_{0}Parcels_7Networks_order_Tian_Subcortex_S{1}_3T_MNI152NLin2009cAsym_{2}mm.nii.gz'.format(n_regions, tian_scale, res)),
            os.path.join(out_dir, 'atlas-MSA{0}_space-MNI152NLin2009cAsym_res-0{1}_dseg.nii.gz'.format(tian_scale, res))
            )

# MNI152NLin6Asym
for tian_scale in [1, 2, 3, 4]:
    out_dir = os.path.join(atlas_dir, 'MSA', 'atlas-MSA{0}'.format(tian_scale))
    for res in [1, 2]:
        shutil.copyfile(
            os.path.join(in_dir, 'Schaefer2018_{0}Parcels_7Networks_order_Tian_Subcortex_S{1}_MNI152NLin6Asym_{2}mm.nii.gz'.format(n_regions, tian_scale, res)),
            os.path.join(out_dir, 'atlas-MSA{0}_space-MNI152NLin6Asym_res-0{1}_dseg.nii.gz'.format(tian_scale, res))
            )

for space in ['MNI152NLin2009cAsym', 'MNI152NLin6Asym']:
    for tian_scale in [1, 2, 3, 4]:
        out_dir = os.path.join(atlas_dir, 'MSA', 'atlas-MSA{0}'.format(tian_scale))
        for res in [1, 2]:
            msa_file = os.path.join(out_dir, 'atlas-MSA{0}_space-{1}_res-0{2}_dseg.nii.gz'.format(tian_scale, space, res))
            msa = nib.load(msa_file)
            msa_data = msa.get_fdata()
            msa_mask = msa_data > msa_max_indices[tian_scale-1]
            msa_data[msa_mask] = 0
            # print(np.unique(msa_data))
            
            # save out (overwrite)
            parc_out = nib.Nifti1Image(msa_data, affine=msa.affine, header=msa.header)
            parc_file_out = os.path.join(out_dir, 'atlas-MSA{0}_space-{1}_res-0{2}_dseg.nii.gz'.format(tian_scale, space, res))
            nib.save(parc_out, parc_file_out)

## Support text files

In [4]:
def process_tsv(tsv_file):
    df = pd.read_csv(tsv_file, header=None)
    idx_filter = np.arange(0, df.shape[0], 2)
    df = df.loc[idx_filter]
    df.reset_index(inplace=True, drop=True)
    df.index = df.index + 1
    df.index.name = 'index'
    df.rename(columns={0: 'label'}, inplace=True)
    
    for i in np.arange(df.shape[0]):
        if '7Networks' in df.loc[i + 1, 'label']:
            df.loc[i + 1, 'cortex'] = True
        else:
            df.loc[i + 1, 'cortex'] = False
    
    df = df.loc[df['cortex'] == False]
    
    df.to_csv(tsv_file, sep="\t")

In [5]:
# copy
in_dir = os.path.join(home, 'MSA/subcortex/Group-Parcellation/3T/Cortex-Subcortex')

for tian_scale in [1, 2, 3, 4]:
    out_dir = os.path.join(atlas_dir, 'MSA', 'atlas-MSA{0}'.format(tian_scale))
    tsv_file = os.path.join(out_dir, 'atlas-MSA{0}_dseg.tsv'.format(tian_scale))

    shutil.copyfile(
        os.path.join(in_dir, 'Schaefer2018_{0}Parcels_7Networks_order_Tian_Subcortex_S{1}_label.txt'.format(n_regions, tian_scale)),
        tsv_file
        )
    
    process_tsv(tsv_file)

    json_file = os.path.join(out_dir, 'atlas-MSA{0}_dseg.json'.format(tian_scale))
    data = {"BIDSVersion": "1.9.0", "Name": "MSA{0}".format(tian_scale)}
    # creating a JSON string
    json_string = json.dumps(data)
    # storing it in a file
    with open(json_file, "w") as json_data:
        json.dump(data, json_data)

## Copy dataset_description

In [6]:
in_file = os.path.join(atlas_dir, 'dataset_description.json')
out_file = os.path.join(atlas_dir, 'MSA', 'dataset_description.json')
shutil.copyfile(in_file, out_file)

'/home/lindenmp/research_projects/snaplab_tools/data/atlases/MSA/dataset_description.json'